# Part 1 of 2: Processing an HTML file


**Exercise ordering:** Each exercise builds logically on previous exercises, but you may solve them in any order. That is, if you can't solve an exercise, you can still move on and try the next one. Use this to your advantage, as the exercises are **not** necessarily ordered in terms of difficulty. Higher point values generally indicate more difficult exercises. 

**Debugging your code:** Right before each exercise test cell, there is a block of text explaining the variables available to you for debugging. You may use these to test your code and can print/display them as needed (careful when printing large objects, you may want to print the head or chunks of rows at a time).

**Exercise point breakdown:**

- Exercise 0: 5 points

**Final reminders:** 

- Submit after **every exercise**
- Review the generated grade report after you submit to see what errors were returned
- Stay calm, skip problems as needed, and take short breaks at your leisure


In [1]:
### Global imports
import dill
from cse6040_devkit import plugins, utils
from cse6040_devkit.training_wheels import run_with_timeout, suppress_stdout
import tracemalloc
from time import time
import hashlib 
import re 
from pprint import pprint



## Topic Introduction

One of the richest sources of information is [the Web](http://www.computerhistory.org/revolution/networking/19/314)! In this notebook, we ask you to use string processing and regular expressions to mine a web page, which is stored in HTML format.

> **Note 0.** The exercises below involve processing of HTML files. However, you don't need to know anything specific about HTML; you can solve (and we have solved) all of these exercises assuming only that the data is a semi-structured string, amenable to simple string manipulation and regular expression processing techniques. In Notebook 6 (optional), you'll see a different method that employs the [Beautiful Soup module](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).
>
> **Note 1.** Following Note 0, there are some outspoken people who believe you should never use regular expressions on HTML. Your instructor finds these arguments to be overly pedantic. For an entertaining take on the subject, see [this blog post](https://blog.codinghorror.com/parsing-html-the-cthulhu-way/).
>
> **Note 2.** The data below is a snapshot from an older version of the Yelp! site. Therefore, you should complete the exercises using the data we've provided, rather than downloading a copy directly from Yelp!.

**The data: Yelp! reviews.** The data you will work with is a snapshot of a recent search on the [Yelp! site](https://yelp.com) for the best fried chicken restaurants in Atlanta. That snapshot is hosted here: https://cse6040.gatech.edu/datasets/yelp-example

If you go ahead and open that site, you'll see that it contains a ranked list of places:

![Top 10 Fried Chicken Spots in ATL as of September 12, 2017](https://cse6040.gatech.edu/datasets/yelp-example/ranked-list-snapshot.png)

**Your task.** In this part of this assignment, we'd like you to write some code to extract this list.

## Getting the data

First things first: you need an HTML file. The following Python code opens a copy of the sample Yelp! page from above.

In [2]:
with open('resource/asnlib/publicdata/yelp.htm', 'r', encoding='utf-8') as f:
    yelp_html = f.read().encode(encoding='utf-8')
    checksum = hashlib.md5(yelp_html).hexdigest()
    assert checksum == "4a74a0ee9cefee773e76a22a52d45a8e", "Downloaded file has incorrect checksum!"
    
print("'yelp.htm' is ready!")

'yelp.htm' is ready!


**Viewing the raw HTML in your web browser.** The file you just downloaded is the raw HTML version of the data described previously. Before moving on, you should go back to that site and use your web browser to view the HTML source for the web page. Do that now to get an idea of what is in that file.

> If you don't know how to view the page source in your browser, try the instructions on [this site](http://www.wikihow.com/View-Source-Code).

**Reading the HTML file into a Python string.** Let's also open the file in Python and read its contents into a string named, `yelp_html`.

In [3]:
with open('resource/asnlib/publicdata/yelp.htm', 'r', encoding='utf-8') as yelp_file:
    yelp_html = yelp_file.read()
    
# Print first few hundred characters of this string:
print("*** type(yelp_html) == {} ***".format(type(yelp_html)))
n = 1000
print("*** Contents (first {} characters) ***\n{} ...".format(n, yelp_html[:n]))

*** type(yelp_html) == <class 'str'> ***
*** Contents (first 1000 characters) ***
<!DOCTYPE html>
<!-- saved from url=(0079)https://www.yelp.com/search?find_desc=fried+chicken&find_loc=Atlanta%2C+GA&ns=1 -->
<html xmlns:fb="http://www.facebook.com/2008/fbml" class="js gr__yelp_com" lang="en"><!--<![endif]--><head data-component-bound="true"><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><link type="text/css" rel="stylesheet" href="./Best Fried chicken in Atlanta, GA - Yelp_files/css"><style type="text/css">.gm-style .gm-style-cc span,.gm-style .gm-style-cc a,.gm-style .gm-style-mtc div{font-size:10px}
</style><style type="text/css">@media print {  .gm-style .gmnoprint, .gmnoprint {    display:none  }}@media screen {  .gm-style .gmnoscreen, .gmnoscreen {    display:none  }}</style><style type="text/css">.gm-style-pbc{transition:opacity ease-in-out;background-color:rgba(0,0,0,0.45);text-align:center}.gm-style-pbt{font-size:22px;color:white;font-family:Roboto,Arial,san

Oy, what a mess! It will be great to have some code read and process the information contained within this file.

### Exercise 0: (5 points)
**extract_ranking**  

**Your task:** define `extract_ranking` as follows:

Create a new function that will return a variable named `rankings`, which is a list of dictionaries set up as follows:

* `rankings[i]` is a dictionary corresponding to the restaurant whose rank is `i+1`. For example, from the screenshot above, `rankings[0]` should be a dictionary with information about Gus's World Famous Fried Chicken.
* Each dictionary, `rankings[i]`, should have these keys:
    * `rankings[i]['name']`: The name of the restaurant, a string.
    * `rankings[i]['stars']`: The star rating, as a string, e.g., `'4.5'`, `'4.0'`
    * `rankings[i]['numrevs']`: The number of reviews, as an **integer.**
    * `rankings[i]['price']`: The price range, as dollar signs, e.g., `'$'`, `'$$'`, `'$$$'`, or `'$$$$'`.

Of course, since the current topic is regular expressions, you might try to apply them (possibly combined with other string manipulation methods) find the particular patterns that yield the desired information.


In [47]:
### Solution - Exercise 0  
def extract_ranking(yelp_html):
    rankings = []
    
    #Extract business names = <a class="biz-name js-analytics-click" data-analytics-label="biz-name"
    #href="https://www.yelp.com/5><span>CalaBar &amp; Grill</span></a>
    
    #Search for non Ads blocks
    organic_section = re.findall(r'<li[^>]*class="regular-search-result"[^>]*>(.*?)</li>', yelp_html, re.DOTALL)
    #html_to_search = organic_section.group(1) if organic_section else yelp_html
    for block in  organic_section:
        if re.search(r'<h3[^>]*>.*?Ad.*?</h3>', block, re.DOTALL):
            continue

        name_match_0 = re.findall(r'<a class="biz-name js-analytics-click"[^>]*><span>(.*?)</span></a>', block)
        name_match = [n for n in name_match_0]
        #print(name_match)

        #Extract stars -  <div class="i-stars i-stars--regular-4 rating-large" title="4.0 star rating">
        stars_match = re.findall(r'<div class="i-stars[^"]*"[^>]*title="([\d\.]+) star rating">', block)
        #print(stars_match)

        #Extract number of reviews - <span class="review-count rating-qualifier">
                #80 reviews
        #</span>
        reviews_match_0 = re.findall(r'<span class="review-count rating-qualifier"[^>]*>\s*([\d]+)\s*reviews\s*</span>',block)
        #print(reviews_match)
        reviews_match = [int(rev) for rev in reviews_match_0]



        #Extract price <span class="business-attribute price-range">$$</span>
        price_match = re.findall(r'<span class="business-attribute price-range[^"]*">([\$]+)</span>', block)
        #print(price_match)

        #print(len(name_match), len(reviews_match), len(price_match), len(stars_match))


        rankings.extend([{"name": n, "numrevs": r, "price": p, "stars": s} for n, r, p, s in zip(name_match, reviews_match, price_match, stars_match)])

    return rankings
        

### Demo function call
demo_ex0_yelp_html = yelp_html
result = extract_ranking(demo_ex0_yelp_html)

print(result)


[{'name': 'Gus’s World Famous Fried Chicken', 'numrevs': 549, 'price': '$$', 'stars': '4.0'}, {'name': 'South City Kitchen - Midtown', 'numrevs': 1777, 'price': '$$', 'stars': '4.5'}, {'name': 'Mary Mac’s Tea Room', 'numrevs': 2241, 'price': '$$', 'stars': '4.0'}, {'name': 'Busy Bee Cafe', 'numrevs': 481, 'price': '$$', 'stars': '4.0'}, {'name': 'Richards’ Southern Fried', 'numrevs': 108, 'price': '$$', 'stars': '4.0'}, {'name': 'Greens &amp; Gravy', 'numrevs': 93, 'price': '$$', 'stars': '3.5'}, {'name': 'Colonnade Restaurant', 'numrevs': 350, 'price': '$$', 'stars': '4.0'}, {'name': 'South City Kitchen Buckhead', 'numrevs': 248, 'price': '$$', 'stars': '4.5'}, {'name': 'Poor Calvin’s', 'numrevs': 1558, 'price': '$$', 'stars': '4.5'}, {'name': 'Rock’s Chicken &amp; Fries', 'numrevs': 67, 'price': '$', 'stars': '4.0'}]


 

**The demo should display this printed output.**
```
[{'name': 'Gus’s World Famous Fried Chicken',
  'numrevs': 549,
  'price': '$$',
  'stars': '4.0'},
 {'name': 'South City Kitchen - Midtown',
  'numrevs': 1777,
  'price': '$$',
  'stars': '4.5'},
 {'name': 'Mary Mac’s Tea Room',
  'numrevs': 2241,
  'price': '$$',
  'stars': '4.0'},
 {'name': 'Busy Bee Cafe', 'numrevs': 481, 'price': '$$', 'stars': '4.0'},
 {'name': 'Richards’ Southern Fried',
  'numrevs': 108,
  'price': '$$',
  'stars': '4.0'},
 {'name': 'Greens &amp; Gravy', 'numrevs': 93, 'price': '$$', 'stars': '3.5'},
 {'name': 'Colonnade Restaurant',
  'numrevs': 350,
  'price': '$$',
  'stars': '4.0'},
 {'name': 'South City Kitchen Buckhead',
  'numrevs': 248,
  'price': '$$',
  'stars': '4.5'},
 {'name': 'Poor Calvin’s', 'numrevs': 1558, 'price': '$$', 'stars': '4.5'},
 {'name': 'Rock’s Chicken &amp; Fries',
  'numrevs': 67,
  'price': '$',
  'stars': '4.0'}]
```


 ---
 <!-- Test Cell Boilerplate -->  
The cell below will test your solution for extract_ranking (exercise 0). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 


In [48]:
### Test Cell - Exercise 0  


from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load
from time import time

tracemalloc.start()
mem_start, peak_start = tracemalloc.get_traced_memory()
print(f"initial memory usage: {mem_start/1024/1024:.2f} MB")

# Load testing utility
with open('resource/asnlib/publicdata/execute_tests', 'rb') as f:
    executor = dill.load(f)

@run_with_timeout(error_threshold=200.0, warning_threshold=100.0)
@suppress_stdout
def execute_tests(**kwargs):
    return executor(**kwargs)


# Execute test
start_time = time()
passed, test_case_vars, e = execute_tests(func=extract_ranking,
              ex_name='extract_ranking',
              key=b'KMsu1cjXXQcBUrrUj5oDFat8HnqoqAILT13DyV66UVg=', 
              n_iter=5)
# Assign test case vars for debugging
input_vars, original_input_vars, returned_output_vars, true_output_vars = test_case_vars
duration = time() - start_time
print(f"Test duration: {duration:.2f} seconds")
current_memory, peak_memory = tracemalloc.get_traced_memory()
print(f"memory after test: {current_memory/1024/1024:.2f} MB")
print(f"memory peak during test: {peak_memory/1024/1024:.2f} MB")
tracemalloc.stop()
if e: raise e
assert passed, 'The solution to extract_ranking did not pass the test.'

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

initial memory usage: 0.00 MB
Test duration: 0.14 seconds
memory after test: 1.47 MB
memory peak during test: 23.73 MB
Passed! Please submit.


**Fin!** This cell marks the end of Part 1. Don't forget to save, restart and rerun all cells, and submit it. When you are done, proceed to Part 2.